In [1]:
from datetime import date, timedelta

import polars as pl

from data_engineering.adsb.read_adsb import read_icaos_from_adsb
from data_engineering.flights.flight_type import get_flights


start_date = date(2026, 3, 1)
end_date = date(2026, 3, 2)
target_dates = [
    start_date + timedelta(days=i)
    for i in range((end_date - start_date).days + 1)
]

df_flights = pl.concat([get_flights(target_date) for target_date in target_dates])
df_ladd_flights = df_flights.filter(pl.col("ladd"))

ladd_icaos = df_ladd_flights.get_column("icao").unique().sort().to_list()
df_adsb = pl.concat([
    read_icaos_from_adsb(target_date, ladd_icaos)
    for target_date in target_dates
])


In [2]:
from flights.flights_match_adsb import flights_match_adsb
matched_icaos = []
for icao in ladd_icaos:
    icao_flights = df_ladd_flights.filter(pl.col("icao") == icao).sort("takeoff_time")
    icao_adsb = df_adsb.filter(pl.col("icao") == icao).sort("time")

    if flights_match_adsb(icao_flights, icao_adsb):
        matched_icaos.append(icao)

df_flights = df_ladd_flights.filter(pl.col("icao").is_in(matched_icaos))

In [ ]:
from pia.simulator.main import simulate_pia_flights
df, truth_dict = simulate_pia_flights(df_flights)

In [4]:
from data_engineering.flights.flight_type import flights_df_to_flights
flights = flights_df_to_flights(df)

In [ ]:
from pia.evaluation.pia_graph_0 import flights_0, test_mapping_dict_0
from pia.evaluation.simulator_flights import flights_real_0
from pia.resolver import build_graph, process_graph
print(f"{len(flights)} nodes")
pia_flights = [flight for flight in flights if flight.pia]
G = build_graph(flights)
# output_path = Path("tmp/pia_graph_0.png")
G, mapping_dict = process_graph(G)
print(f"{len(pia_flights)} pia nodes. {len(mapping_dict)} pia nodes identifed.")

4825 nodes
Removed 1828 nodes
Matched 258
Matched 156
756 pia nodes. 414 pia nodes identifed.


In [6]:
comparison_rows = []
for flight_id in sorted(set(truth_dict) | set(mapping_dict)):
    expected_icao = truth_dict.get(flight_id)
    predicted_icao = mapping_dict.get(flight_id)
    comparison_rows.append({
        "flight_id": flight_id,
        "expected_icao": expected_icao,
        "predicted_icao": predicted_icao,
        "correct": expected_icao == predicted_icao,
        "missing_prediction": expected_icao is not None and predicted_icao is None,
        "extra_prediction": expected_icao is None and predicted_icao is not None,
    })

df_mapping_comparison = pl.DataFrame(comparison_rows)

expected_count = len(truth_dict)
predicted_count = len(mapping_dict)
correct_count = df_mapping_comparison.filter(pl.col("correct")).height

{
    "expected_mappings": expected_count,
    "predicted_mappings": predicted_count,
    "correct_mappings": correct_count,
    "missing_predictions": df_mapping_comparison.filter(pl.col("missing_prediction")).height,
    "extra_predictions": df_mapping_comparison.filter(pl.col("extra_prediction")).height,
    "accuracy": correct_count / expected_count if expected_count else 0,
}


{'expected_mappings': 756,
 'predicted_mappings': 414,
 'correct_mappings': 274,
 'missing_predictions': 342,
 'extra_predictions': 0,
 'accuracy': 0.36243386243386244}

In [8]:
df_mapping_comparison

flight_id,expected_icao,predicted_icao,correct,missing_prediction,extra_prediction
str,str,str,bool,bool,bool
"""pia_00f0b6ed_2026-03-01_23-56""","""a8e78e""","""pia_156d5785""",false,false,false
"""pia_015c0242_2026-03-01_18-53""","""a03427""",null,false,true,false
"""pia_0167f13b_2026-03-02_17-47""","""a74583""",null,false,true,false
"""pia_01a12c7a_2026-03-01_02-55""","""a79e7d""",null,false,true,false
"""pia_01cb8179_2026-03-02_13-11""","""ab7db3""","""ab7db3""",true,false,false
…,…,…,…,…,…
"""pia_fe88bf4d_2026-03-02_14-57""","""a9b3f9""","""ab9dc6""",false,false,false
"""pia_fed89d89_2026-03-01_18-24""","""aae5a5""",null,false,true,false
"""pia_ff317901_2026-03-01_19-17""","""ab4c9e""",null,false,true,false


In [ ]:
df_mapping_comparison

flight_id,expected_icao,predicted_icao,correct,missing_prediction,extra_prediction
str,str,str,bool,bool,bool
"""pia_000453a7_2026-03-01_23-48""","""a33d59""",null,false,true,false
"""pia_000875bb_2026-03-02_14-50""","""a61c8b""",null,false,true,false
"""pia_000896a7_2026-03-02_17-19""","""a2e0a8""","""pia_4754b043""",false,false,false
"""pia_0041212e_2026-03-02_20-34""","""a30f95""",null,false,true,false
"""pia_0088fe05_2026-03-01_22-41""","""a97c74""",null,false,true,false
…,…,…,…,…,…
"""pia_ffc6fd1e_2026-03-01_17-38""","""ac8341""",null,false,true,false
"""pia_ffce60fc_2026-03-02_21-59""","""a46822""",null,false,true,false
"""pia_ffe7650d_2026-03-01_16-12""","""a37a39""",null,false,true,false
